In [10]:
"""
exportar_por_setor.py
=====================
Agrupa o histórico trimestral por setor (SETOR_ATIV do cadastro CVM)
e exporta um Excel com uma aba por setor.
 
Uso:
    from exportar_por_setor import exportar_setores
 
    exportar_setores(historico_completo, "ranking_por_setor.xlsx")
"""
 
import requests
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
 
# ---------------------------------------------------------------------------
# Setores de interesse (na ordem que aparecerão no Excel)
# ---------------------------------------------------------------------------
SETORES = [
    "Máquinas, Equipamentos, Veículos e Peças",
    "Construção Civil, Mat. Constr. e Decoração",
    "Emp. Adm. Participações",
    "Serviços Transporte e Logística",
    "Metalurgia e Siderurgia",
    "Têxtil e Vestuário",
    "Energia Elétrica",
    "Comércio (Atacado e Varejo)",
    "Alimentos",
    "Petroquímicos e Borracha",
    "Telecomunicações",
    "Extração Mineral",
    "Agricultura (Açúcar, Álcool e Cana)",
    "Saneamento, Serv. Água e Gás",
    "Comunicação e Informática",
    # Excluídos intencionalmente:
    # "Securitização de Recebíveis"   → sem ações listadas relevantes
    # "Arrendamento Mercantil"        → financeiras
    "Bancos",   #                        → financeiras
    # "Emp. Adm. Part. - Sem Setor Principal" → muito heterogêneo
    # "Emp. Adm. Part. - Energia Elétrica"    → subsetor de holding
]
 
# Nomes curtos para as abas do Excel (máx. 31 caracteres)
NOME_ABA = {
    "Máquinas, Equipamentos, Veículos e Peças":  "Máquinas e Veículos",
    "Construção Civil, Mat. Constr. e Decoração": "Construção Civil",
    "Emp. Adm. Participações":                    "Holdings",
    "Serviços Transporte e Logística":            "Transporte e Logística",
    "Metalurgia e Siderurgia":                    "Metalurgia",
    "Têxtil e Vestuário":                         "Têxtil e Vestuário",
    "Energia Elétrica":                           "Energia Elétrica",
    "Comércio (Atacado e Varejo)":               "Comércio",
    "Alimentos":                                  "Alimentos",
    "Petroquímicos e Borracha":                   "Petroquímicos",
    "Telecomunicações":                           "Telecomunicações",
    "Extração Mineral":                           "Extração Mineral",
    "Agricultura (Açúcar, Álcool e Cana)":       "Agricultura",
    "Saneamento, Serv. Água e Gás":              "Saneamento e Gás",
    "Comunicação e Informática":                  "Comunicação e TI",
    "Bancos":                                       "Bancos",
}
 
# Colunas a exibir no Excel (na ordem desejada)
COLUNAS_EXIBIR = [
    "DENOM_CIA", "ticker", "DT_FIM_EXERC",
    "ebit_anualizado", "roic",
    "divida_liquida", "pl", "capital_investido",
    "market_cap", "ev", "ev_ebit",
    "rank_roic", "rank_ev_ebit", "rank_magic",
]
 
COLUNAS_RENOMEAR = {
    "DENOM_CIA":       "Empresa",
    "ticker":          "Ticker",
    "DT_FIM_EXERC":    "Trimestre",
    "ebit_anualizado": "EBIT Anualizado (R$)",
    "roic":            "ROIC",
    "divida_liquida":  "Dívida Líquida (R$)",
    "pl":              "Patrim. Líquido (R$)",
    "capital_investido":"Cap. Investido (R$)",
    "market_cap":      "Market Cap (R$)",
    "ev":              "EV (R$)",
    "ev_ebit":         "EV/EBIT",
    "rank_roic":       "Rank ROIC",
    "rank_ev_ebit":    "Rank EV/EBIT",
    "rank_magic":      "Rank Magic",
}
 
# ---------------------------------------------------------------------------
# 1. Mapeamento CNPJ → SETOR_ATIV
# ---------------------------------------------------------------------------
 
def _baixar_setores() -> pd.DataFrame:
    url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
    df  = pd.read_csv(url, sep=";", encoding="latin1", dtype=str)
    df.columns   = df.columns.str.strip()
    df["CNPJ_CIA"]   = df["CNPJ_CIA"].str.strip()
    df["SETOR_ATIV"] = df["SETOR_ATIV"].str.strip()
    return df[["CNPJ_CIA", "SETOR_ATIV"]].drop_duplicates("CNPJ_CIA")
 
 
# ---------------------------------------------------------------------------
# 2. Formatação da aba Excel
# ---------------------------------------------------------------------------
 
def _formatar_aba(ws, n_linhas: int, n_colunas: int) -> None:
    """Aplica cabeçalho, zebra, bordas e larguras."""
    AZUL_HEADER = "1F4E79"
    CINZA_PAR   = "F2F2F2"
    VERDE_ROIC  = "E2EFDA"
    LARANJA_EV  = "FCE4D6"
 
    borda = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"), bottom=Side(style="thin"),
    )
 
    # Cabeçalho
    for col in range(1, n_colunas + 1):
        cell = ws.cell(row=1, column=col)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = PatternFill("solid", start_color=AZUL_HEADER)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = borda
 
    ws.row_dimensions[1].height = 32
 
    # Dados
    for row in range(2, n_linhas + 2):
        fill_cor = CINZA_PAR if row % 2 == 0 else "FFFFFF"
        for col in range(1, n_colunas + 1):
            cell           = ws.cell(row=row, column=col)
            cell.font      = Font(name="Arial", size=9)
            cell.fill      = PatternFill("solid", start_color=fill_cor)
            cell.border    = borda
            cell.alignment = Alignment(horizontal="center", vertical="center")
 
        # Destaque na coluna ROIC (col 5) e EV/EBIT (col 11)
        ws.cell(row=row, column=5).fill  = PatternFill("solid", start_color=VERDE_ROIC)
        ws.cell(row=row, column=11).fill = PatternFill("solid", start_color=LARANJA_EV)
 
    # Larguras por coluna
    larguras = {
        1: 35,   # Empresa
        2: 10,   # Ticker
        3: 12,   # Trimestre
        4: 20,   # EBIT
        5: 10,   # ROIC
        6: 22,   # Dívida Líq.
        7: 22,   # PL
        8: 22,   # Cap. Investido
        9: 20,   # Market Cap
        10: 20,  # EV
        11: 10,  # EV/EBIT
        12: 11,  # Rank ROIC
        13: 13,  # Rank EV/EBIT
        14: 12,  # Rank Magic
    }
    for col, width in larguras.items():
        ws.column_dimensions[get_column_letter(col)].width = width
 
    # Congela cabeçalho
    ws.freeze_panes = "A2"
 
 
# ---------------------------------------------------------------------------
# 3. Função principal
# ---------------------------------------------------------------------------
 
def _rankear_setor(df: pd.DataFrame) -> pd.DataFrame:
    """Recalcula ranks de ROIC, EV/EBIT e Magic DENTRO do setor."""
    df = df.copy()
 
    df["rank_roic_setor"] = (
        df[df["roic"] > 0]
          .groupby("DT_FIM_EXERC")["roic"]
          .rank(ascending=False, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")
 
    df["rank_ev_ebit_setor"] = (
        df[df["ev_ebit"] > 0]
          .groupby("DT_FIM_EXERC")["ev_ebit"]
          .rank(ascending=True, method="min", na_option="bottom")
    ).reindex(df.index).astype("Int64")
 
    r_roic = df["rank_roic_setor"].astype(float)
    r_ev   = df["rank_ev_ebit_setor"].astype(float)
    df["rank_magic_setor"] = np.where(
        r_roic.notna() & r_ev.notna(), r_roic + r_ev, np.nan,
    )
    return df
 
 
def exportar_setores(
    df: pd.DataFrame,
    output_path: str = "ranking_por_setor.xlsx",
) -> None:
    """
    Agrupa o DataFrame histórico por setor e exporta Excel
    com uma aba por setor, com ranks recalculados dentro de cada setor
    e sem duplicatas de ticker por trimestre.
    """
    print("Baixando setores do cadastro CVM...")
    setores_df = _baixar_setores()
    df = df.merge(setores_df, on="CNPJ_CIA", how="left")
    df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])
    print(f"  {df['SETOR_ATIV'].notna().sum():,} linhas com setor mapeado")
 
    rename_setor = {
        **COLUNAS_RENOMEAR,
        "rank_roic_setor":    "Rank ROIC (setor)",
        "rank_ev_ebit_setor": "Rank EV/EBIT (setor)",
        "rank_magic_setor":   "Rank Magic (setor)",
    }
 
    writer = pd.ExcelWriter(output_path, engine="openpyxl")
 
    for setor in SETORES:
        df_setor = df[df["SETOR_ATIV"] == setor].copy()
        if df_setor.empty:
            print(f"  ⚠ Sem dados: {setor}")
            continue
 
        # Remove duplicatas de ticker por trimestre — fica o maior market_cap
        if "ticker" in df_setor.columns:
            com_tk = df_setor[df_setor["ticker"].notna()].copy()
            sem_tk = df_setor[df_setor["ticker"].isna()].copy()
            com_tk = (
                com_tk
                .sort_values("market_cap", ascending=False, na_position="last")
                .drop_duplicates(subset=["ticker", "DT_FIM_EXERC"], keep="first")
            )
            df_setor = pd.concat([com_tk, sem_tk], ignore_index=True)
 
        # Recalcula ranks dentro do setor
        df_setor = _rankear_setor(df_setor)
 
        # Seleciona colunas
        colunas_setor = [
            "CNPJ_CIA", "DENOM_CIA", "ticker", "DT_FIM_EXERC",
            "ebit_anualizado", "roic", "divida_liquida", "pl",
            "capital_investido", "market_cap", "ev", "ev_ebit",
            "rank_roic_setor", "rank_ev_ebit_setor", "rank_magic_setor",
        ]
        cols = [c for c in colunas_setor if c in df_setor.columns]
        df_setor = df_setor[cols].rename(columns=rename_setor)
 
        # Formata
        if "Trimestre" in df_setor.columns:
            df_setor["Trimestre"] = df_setor["Trimestre"].dt.strftime("%Y-%m-%d")
        if "ROIC" in df_setor.columns:
            df_setor["ROIC"] = (df_setor["ROIC"] * 100).round(2)
        for col_r in ["EBIT Anualizado (R$)", "Dívida Líquida (R$)",
                      "Patrim. Líquido (R$)", "Cap. Investido (R$)",
                      "Market Cap (R$)", "EV (R$)"]:
            if col_r in df_setor.columns:
                df_setor[col_r] = df_setor[col_r].round(0)
        if "EV/EBIT" in df_setor.columns:
            df_setor["EV/EBIT"] = df_setor["EV/EBIT"].round(2)
 
        # Ordena: trimestre DESC, rank_magic_setor ASC
        sort_cols, sort_asc = [], []
        if "Trimestre" in df_setor.columns:
            sort_cols.append("Trimestre"); sort_asc.append(False)
        if "Rank Magic (setor)" in df_setor.columns:
            sort_cols.append("Rank Magic (setor)"); sort_asc.append(True)
        if sort_cols:
            df_setor = df_setor.sort_values(sort_cols, ascending=sort_asc,
                                            na_position="last")
 
        nome_aba = NOME_ABA.get(setor, setor[:31])
        df_setor.to_excel(writer, sheet_name=nome_aba, index=False)
        n_emp = df_setor["Ticker"].nunique() if "Ticker" in df_setor.columns else "?"
        print(f"  ✓ {nome_aba}: {len(df_setor):,} linhas ({n_emp} empresas)")
 
    writer.close()
 
    # Aplica formatação visual em todas as abas
    wb = load_workbook(output_path)
    for ws in wb.worksheets:
        n_linhas  = ws.max_row - 1
        n_colunas = ws.max_column
        if n_linhas > 0:
            _formatar_aba(ws, n_linhas, n_colunas)
    wb.save(output_path)
 
    print(f"\n✅ Exportado: {output_path}")
    print(f"   {len(wb.worksheets)} abas | {len(SETORES)} setores solicitados")


In [11]:
    # from adicionar_ev import adicionar_ev
    # historico_completo = adicionar_ev(historico)

ranking_historico = pd.read_csv("ranking.csv")
exportar_setores(ranking_historico, "ranking_por_setor.xlsx")

Baixando setores do cadastro CVM...
  13,606 linhas com setor mapeado
  ✓ Máquinas e Veículos: 615 linhas (13 empresas)
  ✓ Construção Civil: 1,177 linhas (26 empresas)
  ⚠ Sem dados: Emp. Adm. Participações
  ✓ Transporte e Logística: 661 linhas (14 empresas)
  ✓ Metalurgia: 386 linhas (12 empresas)
  ✓ Têxtil e Vestuário: 605 linhas (14 empresas)
  ✓ Energia Elétrica: 903 linhas (12 empresas)
  ✓ Comércio: 1,116 linhas (27 empresas)
  ✓ Alimentos: 347 linhas (5 empresas)
  ✓ Petroquímicos: 157 linhas (3 empresas)
  ✓ Telecomunicações: 234 linhas (5 empresas)
  ✓ Extração Mineral: 85 linhas (3 empresas)
  ✓ Agricultura: 304 linhas (10 empresas)
  ✓ Saneamento e Gás: 231 linhas (4 empresas)
  ✓ Comunicação e TI: 446 linhas (10 empresas)
  ✓ Bancos: 329 linhas (13 empresas)

✅ Exportado: ranking_por_setor.xlsx
   15 abas | 16 setores solicitados
